In [ ]:
import chromadb

print("1. Initializing the Database Engine...")
# We use a PersistentClient so the database is saved to a folder named 'cyber_brain'.
# If the script is stopped and run again tomorrow, the data will still be there.
client = chromadb.PersistentClient(path="./cyber_brain")

print("2. Creating the Collection (The Filing Cabinet)...")
# get_or_create prevents the script from crashing if the collection already exists.
collection = client.get_or_create_collection(name="mitre_microscopic_dataset")

print("3. Defining the Threat Intelligence (The MITRE Techniques)...")
# We are manually hardcoding our threat intelligence dataset. 
# In a real pipeline, we'll use a JSON file or API.

mitre_data = [
    {
        "id": "T1059",
        "name": "Command and Scripting Interpreter",
        "tactic": "Execution",
        "description": "Adversaries may abuse command and script interpreters to execute commands, scripts, or binaries. These interfaces and languages provide ways of interacting with computer systems and are a common feature across many different platforms. For example, attackers might use PowerShell or Windows Command Shell to run malicious code."
    },
    {
        "id": "T1566",
        "name": "Phishing",
        "tactic": "Initial Access",
        "description": "Adversaries may send phishing messages to gain access to victim systems. All forms of phishing are electronically delivered social engineering. Phishing can be targeted, known as spearphishing. In spearphishing, a specific individual, company, or industry will be targeted with information tailored to them to open malicious attachments or links."
    },
    {
        "id": "T1078",
        "name": "Valid Accounts",
        "tactic": "Defense Evasion",
        "description": "Adversaries may obtain and abuse credentials of existing accounts as a means of gaining Initial Access, Persistence, Privilege Escalation, or Defense Evasion. Compromised credentials may be used to bypass access controls placed on various resources on systems within the network and may even be used for persistent access to remote systems."
    }
]

print("4. Formatting the data for ChromaDB...")
# ChromaDB's .add() method requires strict list formatting.
# We must split our dictionaries into three separate, aligned lists.
document_list = []
metadata_list = []
id_list = []

for technique in mitre_data:
    # The document is the raw text the RAG assistant will actually "read" and embed
    document_list.append(technique["description"])
    
    # Metadatas are facts we can filter by later
    metadata_list.append({"technique_id": technique["id"], "name": technique["name"], "tactic": technique["tactic"]})
    
    # IDs must be unique strings
    id_list.append(technique["id"])

print("5. Ingesting the data (Generating Embeddings and Inserting)...")
# This is where the magic happens. By passing the documents, ChromaDB automatically 
# downloads a sentence-transformer model in the background, converts the raw text 
# into high-dimensional vectors, and saves them to the local folder.
collection.upsert(
    documents=document_list,
    metadatas=metadata_list,
    ids=id_list
)
# Note: I used .upsert() instead of .add(). Upsert means "Update or Insert". 
# It prevents errors if the script is run twice and try to add identical IDs.

print("\n--- INGESTION COMPLETE ---\n")

print("6. Querying the Database (Testing the RAG Retrieval)...")
# We ask a natural language question. Notice how our query does NOT contain 
# the exact words "T1566" or "Phishing".
question = "How do hackers trick employees into clicking bad links to get inside the network?"

print(f"Question: '{question}'\n")

# We ask Chroma to embed our question and find the 1 closest vector (n_results=1)
results = collection.query(
    query_texts=[question],
    n_results=1
)

# Extracting the messy dictionary response
retrieved_id = results['ids'][0][0]
retrieved_name = results['metadatas'][0][0]['name']
retrieved_desc = results['documents'][0][0]

print(f"Top Match Found: {retrieved_id} - {retrieved_name}")
print(f"Description: {retrieved_desc}")